Install Packages

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [2]:
import os

print("Current directory:")
print(os.getcwd())

print("\nFiles/folders:")
!ls -lah

Current directory:
/content

Files/folders:
total 16K
drwxr-xr-x 1 root root 4.0K Sep  4 13:32 .
drwxr-xr-x 1 root root 4.0K Sep 17 15:47 ..
drwxr-xr-x 4 root root 4.0K Sep  4 13:32 .config
drwxr-xr-x 1 root root 4.0K Sep  4 13:32 sample_data


In [ ]:
# Install packages
# %pip install -r ../../requirements.txt

In [ ]:
# Install scispacy model for classification (inside .venv)
# pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

In [ ]:
import datasets
import pandas as pd
import numpy as np
import spacy
import scispacy
import torch
import peft
import accelerate
import bitsandbytes as bnb
from transformers import pipeline

print("datasets:", datasets.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("spacy:", spacy.__version__)
print("scispacy:", scispacy.__version__)
print("torch:", torch.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("bitsandbytes:", bnb.__version__)

/Users/chris/Desktop/Research/recursive-llm-degradation-research/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


datasets: 5.0.1
pandas: 3.0.5
numpy: 1.26.4
spacy: 3.7.5
scispacy: 0.6.2
torch: 2.14.0
peft: 0.21.0
accelerate: 1.15.0
bitsandbytes: 0.50.2


In [ ]:
from datasets import load_dataset

unlabeled_data = load_dataset("qiaojin/PubMedQA", "pqa_unlabeled", split="train")
labeled_data = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")

In [ ]:
print(unlabeled_data)
print(unlabeled_data[0])
print(labeled_data)
print(labeled_data[0])

Dataset({
    features: ['pubid', 'question', 'context', 'long_answer'],
    num_rows: 61249
})
{'pubid': 14499029, 'question': 'Is naturopathy as effective as conventional therapy for treatment of menopausal symptoms?', 'context': {'contexts': ['Although the use of alternative medicine in the United States is increasing, no published studies have documented the effectiveness of naturopathy for treatment of menopausal symptoms compared to women receiving conventional therapy in the clinical setting.', 'To compare naturopathic therapy with conventional medical therapy for treatment of selected menopausal symptoms.', 'A retrospective cohort study, using abstracted data from medical charts.', 'One natural medicine and six conventional medical clinics at Community Health Centers of King County, Washington, from November 1, 1996, through July 31, 1998.', 'Women aged 40 years of age or more with a diagnosis of menopausal symptoms documented by a naturopathic or conventional physician.', 'Imp

In [ ]:
# Before using the labeled set as your gold evaluation set, check whether its questions/pubids overlap with the unlabeled pool. If there is overlap, we must exclude those rows from your training pool.
labeled_ids = set(labeled_data["pubid"])
unlabeled_ids = set(unlabeled_data["pubid"])

overlap = labeled_ids.intersection(unlabeled_ids)

print("Overlap:", len(overlap))
gold_eval_set = labeled_data
# Remove any gold-eval examples from the unlabeled training pool
if len(overlap) > 0:
    unlabeled_data = unlabeled_data.filter(
        lambda row: row["pubid"] not in overlap
    )
print(f"Gold eval: {len(gold_eval_set)}")

Overlap: 0
Gold eval: 1000


In [ ]:
# Create human training pool
SEED = 42

unlabeled_data_shuffled = unlabeled_data.shuffle(seed=SEED)

G0_BASE_SIZE = 500
ANCHOR_POOL_SIZE = 500

g0_human_train = unlabeled_data_shuffled.select(range(G0_BASE_SIZE))

human_anchor_pool = unlabeled_data_shuffled.select(
    range(G0_BASE_SIZE, G0_BASE_SIZE + ANCHOR_POOL_SIZE)
)

print(f"G0 train: {len(g0_human_train)}")
print(f"Anchor pool: {len(human_anchor_pool)}")
print(f"Gold eval: {len(gold_eval_set)}")

G0 train: 500
Anchor pool: 500
Gold eval: 1000


In [ ]:
print(g0_human_train[0])

{'pubid': 22949290, 'question': 'Unipolar mania: a distinct entity or characteristic of manic preponderance?', 'context': {'contexts': ['It has been reported that fewer patients with unipolar mania respond to lithium prophylaxis as do those with classical bipolar disorder. This study aimed to determine if the difference to response to lithium is related to unipolar mania or to a high preponderance of mania during the course of bipolarity.', 'The study included bipolar-I patients (according to DSM-IV criteria) that had a ≥ 2-year history of either lithium or valproate prophylaxis as monotherapy. The response rate in the patients with unipolar mania and classical bipolar disorder were compared. Then, the response rate to lithium in all the patients with a manic episode rate<50% and>50%, and<80% and>80% during their course were compared. Finally, the above comparisons were repeated, excluding the patients with unipolar mania.', 'The study included 121 bipolar-I patients (34 unipolar mania

In [ ]:
# Prompt formatting
def format_pubmed_qa(row):
    ctx = row["context"]
    if isinstance(ctx, dict) and "contexts" in ctx:
        context_str = " ".join(ctx["contexts"])
    elif isinstance(ctx, list):
        context_str = " ".join(ctx)
    else:
        context_str = str(ctx)
    return {
        "text": (
            f"### Question:\n{row['question']}\n\n"
            f"### Context:\n{context_str}\n\n"
            f"### Long Answer:\n{row['long_answer']}"
        )
    }

g0_formatted = g0_human_train.map(format_pubmed_qa)
print("Number of formatted rows:", len(g0_formatted))
print("\n--- SAMPLE ---\n")
print(g0_formatted[0]["text"])

Map: 100%|██████████| 500/500 [00:00<00:00, 7862.98 examples/s]

Number of formatted rows: 500

--- SAMPLE ---

### Question:
Unipolar mania: a distinct entity or characteristic of manic preponderance?

### Context:
It has been reported that fewer patients with unipolar mania respond to lithium prophylaxis as do those with classical bipolar disorder. This study aimed to determine if the difference to response to lithium is related to unipolar mania or to a high preponderance of mania during the course of bipolarity. The study included bipolar-I patients (according to DSM-IV criteria) that had a ≥ 2-year history of either lithium or valproate prophylaxis as monotherapy. The response rate in the patients with unipolar mania and classical bipolar disorder were compared. Then, the response rate to lithium in all the patients with a manic episode rate<50% and>50%, and<80% and>80% during their course were compared. Finally, the above comparisons were repeated, excluding the patients with unipolar mania. The study included 121 bipolar-I patients (34 unipol

In [ ]:
print(g0_formatted[0])

{'pubid': 22949290, 'question': 'Unipolar mania: a distinct entity or characteristic of manic preponderance?', 'context': {'contexts': ['It has been reported that fewer patients with unipolar mania respond to lithium prophylaxis as do those with classical bipolar disorder. This study aimed to determine if the difference to response to lithium is related to unipolar mania or to a high preponderance of mania during the course of bipolarity.', 'The study included bipolar-I patients (according to DSM-IV criteria) that had a ≥ 2-year history of either lithium or valproate prophylaxis as monotherapy. The response rate in the patients with unipolar mania and classical bipolar disorder were compared. Then, the response rate to lithium in all the patients with a manic episode rate<50% and>50%, and<80% and>80% during their course were compared. Finally, the above comparisons were repeated, excluding the patients with unipolar mania.', 'The study included 121 bipolar-I patients (34 unipolar mania

Load PubMedBERT BC5CDR NER

In [ ]:
# Try new classifier
pubmed_ner_pipeline = pipeline(
    "token-classification",
    model="HFpf/pubmedbert-bc5cdr-ner",
    aggregation_strategy="simple"
)

print("PubMedBERT BC5CDR NER loaded!")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6124.06it/s]


PubMedBERT BC5CDR NER loaded!


In [ ]:
# B- (Begin) — marks the first token of an entity span.
# I- (Inside) — marks any subsequent token that continues the same entity span.
# O (Outside) — the token isn't part of any entity at all.
LABEL_MAP = {
    "LABEL_0": "O",
    "LABEL_1": "B-Chemical",
    "LABEL_2": "B-Disease",
    "LABEL_3": "I-Disease",
    "LABEL_4": "I-Chemical"
}

def extract_pubmed_entities(text):
    raw_entities = pubmed_ner_pipeline(text)

    entities = []
    current = None

    for entity in raw_entities:
        label = LABEL_MAP[entity["entity_group"]]

        # Outside any entity
        if label == "O":
            if current is not None:
                entities.append(current)
                current = None
            continue

        prefix, entity_type = label.split("-", 1)

        start = entity["start"]
        end = entity["end"]
        score = float(entity["score"])

        # Beginning of a new entity
        if prefix == "B":
            if current is not None:
                entities.append(current)

            current = {
                "entity": text[start:end],
                "type": entity_type,
                "start": start,
                "end": end,
                "scores": [score]
            }

        # Continuation of current entity
        elif prefix == "I":
            if current is not None and current["type"] == entity_type:
                current["entity"] = text[current["start"]:end]
                current["end"] = end
                current["scores"].append(score)

    if current is not None:
        entities.append(current)

    # Average scores across entity pieces
    for entity in entities:
        entity["score"] = (
            sum(entity["scores"]) / len(entity["scores"])
        )
        del entity["scores"]

    return entities

In [ ]:
text = (
    "Patients with bipolar disorder were treated "
    "with lithium and valproate."
)
print(g0_formatted[0]["text"])

entities = extract_pubmed_entities(text)

for entity in entities:
    print(
        f"{entity['type']:10s} | "
        f"{entity['entity']:25s} | "
        f"score={entity['score']:.3f}"
    )

### Question:
Unipolar mania: a distinct entity or characteristic of manic preponderance?

### Context:
It has been reported that fewer patients with unipolar mania respond to lithium prophylaxis as do those with classical bipolar disorder. This study aimed to determine if the difference to response to lithium is related to unipolar mania or to a high preponderance of mania during the course of bipolarity. The study included bipolar-I patients (according to DSM-IV criteria) that had a ≥ 2-year history of either lithium or valproate prophylaxis as monotherapy. The response rate in the patients with unipolar mania and classical bipolar disorder were compared. Then, the response rate to lithium in all the patients with a manic episode rate<50% and>50%, and<80% and>80% during their course were compared. Finally, the above comparisons were repeated, excluding the patients with unipolar mania. The study included 121 bipolar-I patients (34 unipolar mania and 87 classical bipolar disorder). Th

In [ ]:
for i in range(10):
    text = g0_formatted[i]["long_answer"]
    print(text)
    print(f"\n{'=' * 80}")
    print(f"Example {i}")
    print(f"{'=' * 80}")

    entities = extract_pubmed_entities(text)

    for entity in entities:
        print(
            f"{entity['type']:10s} | "
            f"{entity['entity']:35s} | "
            f"score={entity['score']:.3f}"
        )

Fewer patients with unipolar mania responded to lithium prophylaxis than those with classical bipolar disorder, which appeared to be related to unipolar mania, rather than to a high manic predominance during the disease course. On the other hand, response to valproate prophylaxis was similar in the unipolar mania and classical bipolar disorder groups.

Example 0
Disease    | unipolar mania                      | score=0.985
Chemical   | lithium                             | score=1.000
Disease    | bipolar disorder                    | score=0.999
Disease    | unipolar mania                      | score=0.978
Disease    | manic                               | score=0.661
Chemical   | valproate                           | score=1.000
Disease    | unipolar mania                      | score=0.988
Disease    | bipolar disorder                    | score=0.999
Our findings demonstrate socioeconomic inequalities in AMI incidence in migrant groups and suggest a convergence towards the Dutch 

In [ ]:
from transformers import AutoTokenizer
import numpy as np

MODEL_NAME = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

token_lengths = []

for row in g0_formatted:
    token_ids = tokenizer(
        row["text"],
        add_special_tokens=True
    )["input_ids"]

    token_lengths.append(len(token_ids))

In [ ]:
print("Min:", np.min(token_lengths))
print("Mean:", np.mean(token_lengths))
print("Median:", np.median(token_lengths))
print("90th percentile:", np.percentile(token_lengths, 90))
print("95th percentile:", np.percentile(token_lengths, 95))
print("99th percentile:", np.percentile(token_lengths, 99))
print("Max:", np.max(token_lengths))

Min: 157
Mean: 407.664
Median: 395.5
90th percentile: 531.0
95th percentile: 602.1999999999998
99th percentile: 801.0799999999999
Max: 1126


In [ ]:
for limit in [512, 768, 1024, 1536, 2048]:
    count = sum(length > limit for length in token_lengths)

    print(
        f"> {limit}: "
        f"{count} examples "
        f"({count / len(token_lengths) * 100:.2f}%)"
    )

> 512: 64 examples (12.80%)
> 768: 7 examples (1.40%)
> 1024: 2 examples (0.40%)
> 1536: 0 examples (0.00%)
> 2048: 0 examples (0.00%)


In [ ]:
# Use 768 as 99th percentile is  733
long_indices = [
    i for i, length in enumerate(token_lengths)
    if length > 1024
]

print("Examples over 768:", len(long_indices))

for i in long_indices[:5]:
    print("=" * 80)
    print("Index:", i)
    print("Tokens:", token_lengths[i])
    print()
    print(g0_formatted[i]["text"][:1500])
    print()

Examples over 768: 2
Index: 45
Tokens: 1103

### Question:
Is Pes Cavus Alignment Associated With Lisfranc Injuries of the Foot?

### Context:
Lisfranc (tarsometatarsal joint) injuries are relatively rare, accounting for less than 1% of all fractures, and as many as 20% of subtle Lisfranc injuries are missed at the initial patient presentation. An undiagnosed Lisfranc injury can have devastating consequences to the patient. Therefore, any factor that can raise a clinician's index of suspicion to make this diagnosis is potentially important. The cavus foot has been associated with various maladies of the lower extremity, but to our knowledge, it has not been reported to be associated with Lisfranc injury.QUESTIONS/ Do patients who experience a low-energy Lisfranc injury have greater talar head coverage and a greater talo-first metatarsal angle than control subjects? A retrospective, case-control study was conducted from September 2011 to December 2014 to identify patients diagnosed and 

Tokenization

Only use long_answer for loss not quesiton and context

In [ ]:
MAX_LENGTH = 1024

def tokenize_function(row):
    context = row["context"]

    if isinstance(context, dict) and "contexts" in context:
        context_text = " ".join(context["contexts"])
    elif isinstance(context, list):
        context_text = " ".join(context)
    else:
        context_text = str(context)

    prompt = (
        f"### Question:\n{row['question']}\n\n"
        f"### Context:\n{context_text}\n\n"
        f"### Long Answer:\n"
    )

    answer = row["long_answer"]

    prompt_tokens = tokenizer(
        prompt,
        add_special_tokens=False
    )["input_ids"]

    answer_tokens = tokenizer(
        answer,
        add_special_tokens=False
    )["input_ids"]

    input_ids = prompt_tokens + answer_tokens
    # -100 means ignore token when calculating training loss
    # Positive numbers are actual token IDs
    labels = (
        [-100] * len(prompt_tokens)
        + answer_tokens
    )

    input_ids = input_ids[:MAX_LENGTH]
    labels = labels[:MAX_LENGTH]

    attention_mask = [1] * len(input_ids)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [ ]:
g0_tokenized = g0_formatted.map(
    tokenize_function,
    batched=False
)

Map: 100%|██████████| 500/500 [00:00<00:00, 1043.12 examples/s]


In [ ]:
row = g0_tokenized[0]

print("input_ids length:", len(row["input_ids"]))
print("labels length:", len(row["labels"]))
print("attention_mask length:", len(row["attention_mask"]))

input_ids length: 402
labels length: 402
attention_mask length: 402


In [ ]:
row = g0_tokenized[0]

ignored = row["labels"].count(-100)

trained = sum(
    label != -100
    for label in row["labels"]
)

print("Ignored prompt tokens (Question + Context):", ignored)
print("Answer tokens used for loss (Answer):", trained)

Ignored prompt tokens (Question + Context): 327
Answer tokens used for loss (Answer): 75


In [ ]:
answer_token_ids = [
    label
    for label in row["labels"]
    if label != -100
]

decoded_answer = tokenizer.decode(
    answer_token_ids,
    skip_special_tokens=True
)

print(decoded_answer)

Fewer patients with unipolar mania responded to lithium prophylaxis than those with classical bipolar disorder, which appeared to be related to unipolar mania, rather than to a high manic predominance during the disease course. On the other hand, response to valproate prophylaxis was similar in the unipolar mania and classical bipolar disorder groups.


In [ ]:
print(g0_formatted[0]["long_answer"])

Fewer patients with unipolar mania responded to lithium prophylaxis than those with classical bipolar disorder, which appeared to be related to unipolar mania, rather than to a high manic predominance during the disease course. On the other hand, response to valproate prophylaxis was similar in the unipolar mania and classical bipolar disorder groups.


QLoRA Setup

In [ ]:
import torch
import transformers
import peft
import accelerate
import bitsandbytes as bnb

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("bitsandbytes:", bnb.__version__)

print()
print("MPS available:", torch.backends.mps.is_available())

torch: 2.14.0
transformers: 5.17.0
peft: 0.21.0
accelerate: 1.15.0
bitsandbytes: 0.50.2

MPS available: True


In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using:", device)


Using: mps


In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Config might change for CUDA if GPU supports BF16
# Ex: bnb_4bit_compute_dtype=torch.bfloat16
'''
Load Qwen using 4-bit weights
        ↓
Use NF4 representation
        ↓
Do calculations in FP16
        ↓
Apply extra compression to quantization metadata
'''
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded successfully.")
print("Model device:", next(model.parameters()).device)

Loading weights: 100%|██████████| 290/290 [00:03<00:00, 94.23it/s] 


Model loaded successfully.
Model device: mps:0


Prepare model for QLoRA training

In [ ]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

print("Model prepared for k-bit training.")

Model prepared for k-bit training.


In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
from peft import get_peft_model

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [ ]:
row = g0_tokenized[0]

print("First 40 labels:")
print(row["labels"][:40])

print("\nLast 40 labels:")
print(row["labels"][-40:])
print(len(row["labels"]))

First 40 labels:
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]

Last 40 labels:
[1091, 311, 264, 1550, 69122, 33964, 681, 2337, 279, 8457, 3308, 13, 1913, 279, 1008, 1424, 11, 2033, 311, 1044, 776, 349, 39234, 3923, 7184, 572, 4428, 304, 279, 650, 573, 7417, 883, 685, 323, 28824, 64819, 19267, 5203, 13]
402


In [ ]:
def custom_data_collator(features):
    max_length = max(len(f["input_ids"]) for f in features)

    input_ids = []
    attention_masks = []
    labels = []

    for feature in features:
        pad_length = max_length - len(feature["input_ids"])

        input_ids.append(
            feature["input_ids"] + [tokenizer.pad_token_id] * pad_length
        )

        attention_masks.append(
            feature["attention_mask"] + [0] * pad_length
        )

        labels.append(
            feature["labels"] + [-100] * pad_length
        )

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=f"../models/{MODEL_NAME}-g0",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,
    bf16=True,
    gradient_checkpointing=True,
    torch_empty_cache_steps=50,
    report_to="none",
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=g0_tokenized,
    data_collator=custom_data_collator
)

In [ ]:
print("Trainer created successfully.")
print("Training examples:", len(g0_tokenized))
print("Device:", trainer.args.device)

Trainer created successfully.
Training examples: 500
Device: mps


In [ ]:
train_dataloader = trainer.get_train_dataloader()

batch = next(iter(train_dataloader))

print(batch.keys())
print("input_ids shape:", batch["input_ids"].shape)
print("attention_mask shape:", batch["attention_mask"].shape)
print("labels shape:", batch["labels"].shape)

dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids shape: torch.Size([1, 416])
attention_mask shape: torch.Size([1, 416])
labels shape: torch.Size([1, 416])


/Users/chris/Desktop/Research/recursive-llm-degradation-research/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
print("input_ids device:", batch["input_ids"].device)
print("Model device:", next(model.parameters()).device)

input_ids device: mps:0
Model device: mps:0


In [ ]:
print("First 30 input IDs:")
print(batch["input_ids"][0][:30])

print("\nFirst 30 labels:")
print(batch["labels"][0][:30])

First 30 input IDs:
tensor([14374, 15846,   510, 21468,  1356,  2851,   278, 62759, 32587,  8063,
        23460, 46347, 98547,  7269,  9387, 17984,  7813,   304,  6835,   448,
        46347, 18906, 81989,  5866,   315,   220,    19,    13,    16,    12],
       device='mps:0')

First 30 labels:
tensor([-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100], device='mps:0')


In [ ]:
padding_count = (batch["labels"] == -100).sum().item()

print("Ignored/padded label positions:", padding_count)

Ignored/padded label positions: 317


In [ ]:
trainer.train()
model.save_pretrained(F"../models/{MODEL_NAME}-g0")
tokenizer.save_pretrained(F"../models/{MODEL_NAME}-g0")

Step,Training Loss
10,2.412893
20,2.336673
30,2.453428
40,2.376309
50,2.308429
60,2.317807


('./models/qwen2.5-0.5b-g0/tokenizer_config.json',
 './models/qwen2.5-0.5b-g0/chat_template.jinja',
 './models/qwen2.5-0.5b-g0/tokenizer.json')

In [ ]:
# smoke_train = g0_tokenized.select(range(100))

# print("Smoke-test examples:", len(smoke_train))

In [ ]:
# trainer.train()
# model.save_pretrained("../models/qwen2.5-0.5b-g0")
# tokenizer.save_pretrained("../models/qwen2.5-0.5b-g0")

In [ ]:
# from transformers import Trainer

# smoke_trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=smoke_train,
#     data_collator=custom_data_collator
# )

In [ ]:
# smoke_trainer.train()

In [ ]:
# Testing the model
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

from peft import PeftModel
ADAPTER_PATH = f"../models/{MODEL_NAME}-g0"

tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_PATH
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using:", device)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map=device
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

model.eval()
print("Model loaded.")
print("Device:", next(model.parameters()).device)

for i in range(3):
    row = gold_eval_set[i]

    context = row["context"]

    if isinstance(context, dict) and "contexts" in context:
        context_text = " ".join(context["contexts"])
    elif isinstance(context, list):
        context_text = " ".join(context)
    else:
        context_text = str(context)

    prompt = (
        f"### Question:\n{row['question']}\n\n"
        f"### Context:\n{context_text}\n\n"
        f"### Long Answer:\n"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    prompt_length = inputs["input_ids"].shape[1]

    generated_ids = output_ids[0][prompt_length:]

    generated_answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    print("\n" + "=" * 100)
    print(f"EXAMPLE {i}")
    print("=" * 100)

    print("\nQUESTION:")
    print(row["question"])

    print("\nGENERATED ANSWER:")
    print(generated_answer)

    print("\nHUMAN ANSWER:")
    print(row["long_answer"])

Using: mps


Loading weights: 100%|██████████| 290/290 [00:03<00:00, 88.68it/s] 


Model loaded.
Device: mps:0

EXAMPLE 0

QUESTION:
Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?

GENERATED ANSWER:
Mitochondria may be involved in the regulation of PPD in Aponothea madagassariensis by influencing the rate of PDC. These findings suggest that PCD can occur without the involvement of mitochondrion. Further studies should examine whether this process could be used as a tool for monitoring PCD or other developmental events in plants.
This work provides evidence suggesting that PCC does not require the participation of mitochondrii. It suggests that PDC involves multiple mechanisms, which include changes in mitochondrial function. Future research should focus on investigating how PDC affects the structure of the vascular

HUMAN ANSWER:
Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle with other organelles during developmental PCD. To the best 

# Evaluation is doing from here


In [3]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!find /content/drive -type d -name "Qwen2.5-0.5B-g0"

/content/drive/.shortcut-targets-by-id/1eQJTWimEKgHUoLDgs8cXZgM38VRJP20l/Qwen2.5-0.5B-g0
/content/drive/.Encrypted/.shortcut-targets-by-id/1eQJTWimEKgHUoLDgs8cXZgM38VRJP20l/Qwen2.5-0.5B-g0


In [5]:
G0_PATH = "/content/drive/.shortcut-targets-by-id/1eQJTWimEKgHUoLDgs8cXZgM38VRJP20l/Qwen2.5-0.5B-g0"

!ls -lah "$G0_PATH"

total 45M
drwx------ 4 root root 4.0K Sep 17 15:16 .
dr-x------ 3 root root 4.0K Sep 17 15:16 ..
-rw------- 1 root root 1.2K Sep 17 14:41 adapter_config.json
-rw------- 1 root root  34M Sep 17 14:41 adapter_model.safetensors
-rw------- 1 root root 2.4K Sep 17 14:41 chat_template.jinja
drwx------ 2 root root 4.0K Sep 17 15:16 checkpoint-2
-rw------- 1 root root 5.1K Sep 17 14:41 README.md
-rw------- 1 root root  697 Sep 17 14:41 tokenizer_config.json
-rw------- 1 root root  11M Sep 17 14:41 tokenizer.json


In [6]:
import json

G0_PATH = "/content/drive/.shortcut-targets-by-id/1eQJTWimEKgHUoLDgs8cXZgM38VRJP20l/Qwen2.5-0.5B-g0"

with open(f"{G0_PATH}/adapter_config.json", "r") as f:
    adapter_config = json.load(f)

print(json.dumps(adapter_config, indent=2))

{
  "alora_invocation_tokens": null,
  "alpha_pattern": {},
  "arrow_config": null,
  "auto_mapping": null,
  "base_model_name_or_path": "Qwen/Qwen2.5-0.5B",
  "bias": "none",
  "corda_config": null,
  "ensure_weight_tying": false,
  "eva_config": null,
  "exclude_modules": null,
  "fan_in_fan_out": false,
  "inference_mode": true,
  "init_lora_weights": true,
  "kasa_config": null,
  "layer_replication": null,
  "layers_pattern": null,
  "layers_to_transform": null,
  "loftq_config": {},
  "lora_alpha": 32,
  "lora_bias": false,
  "lora_dropout": 0.05,
  "lora_ga_config": null,
  "megatron_config": null,
  "megatron_core": "megatron.core",
  "modules_to_save": null,
  "monteclora_config": null,
  "peft_type": "LORA",
  "peft_version": "0.21.0",
  "qalora_group_size": 16,
  "r": 16,
  "rank_pattern": {},
  "revision": null,
  "target_modules": [
    "v_proj",
    "gate_proj",
    "down_proj",
    "up_proj",
    "o_proj",
    "q_proj",
    "k_proj"
  ],
  "target_parameters": null,
  "t

In [9]:
from datasets import load_dataset

labeled_data = load_dataset(
    "qiaojin/PubMedQA",
    "pqa_labeled",
    split="train"
)

gold_eval_set = labeled_data

print("Gold evaluation set loaded!")
print("Number of examples:", len(gold_eval_set))

README.md:   0%|          | 0.00/5.19k [00:00<?, ?B/s]

pqa_labeled/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.08MB            

pqa_labeled/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Gold evaluation set loaded!
Number of examples: 1000


In [7]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-0.5B"

G0_PATH = "/content/drive/.shortcut-targets-by-id/1eQJTWimEKgHUoLDgs8cXZgM38VRJP20l/Qwen2.5-0.5B-g0"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(G0_PATH)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Loading G0 LoRA adapter...")

g0_model = PeftModel.from_pretrained(
    base_model,
    G0_PATH
)

g0_model.eval()

print("G0 model loaded successfully!")

Loading tokenizer...
Loading base model...


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Loading G0 LoRA adapter...


/usr/local/lib/python3.13/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['kasa_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


G0 model loaded successfully!


In [10]:
# Test G0 generation on 3 examples

import torch

g0_model.eval()

for i in range(3):
    row = gold_eval_set[i]

    context = row["context"]

    if isinstance(context, dict) and "contexts" in context:
        context_text = " ".join(context["contexts"])
    elif isinstance(context, list):
        context_text = " ".join(context)
    else:
        context_text = str(context)

    prompt = (
        f"### Question:\n{row['question']}\n\n"
        f"### Context:\n{context_text}\n\n"
        f"### Long Answer:\n"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    inputs = {k: v.to(g0_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = g0_model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    prompt_length = inputs["input_ids"].shape[1]

    generated_ids = output_ids[0][prompt_length:]

    generated_answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    print("=" * 80)
    print(f"Example {i + 1}")
    print("QUESTION:")
    print(row["question"])
    print("\nGOLD ANSWER:")
    print(row["long_answer"])
    print("\nG0 ANSWER:")
    print(generated_answer)

Example 1
QUESTION:
Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?

GOLD ANSWER:
Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle with other organelles during developmental PCD. To the best of our knowledge, this is the first report of mitochondria and chloroplasts moving on transvacuolar strands to form a ring structure surrounding the nucleus during developmental PCD. Also, for the first time, we have shown the feasibility for the use of CsA in a whole plant system. Overall, our findings implicate the mitochondria as playing a critical and early role in developmentally regulated PCD in the lace plant.

G0 ANSWER:
Mitochondria have long been known as important regulators of cellular function. They provide energy for many processes such as growth, division, and apoptosis. However, their roles in developmental events remain poorly understood. In this study,

In [14]:
print("Padding side:", tokenizer.padding_side)
print("Pad token:", tokenizer.pad_token)
print("Pad token ID:", tokenizer.pad_token_id)
print("EOS token ID:", tokenizer.eos_token_id)

Padding side: right
Pad token: <|endoftext|>
Pad token ID: 151643
EOS token ID: 151643


In [15]:
# Set left padding for batched generation
tokenizer.padding_side = "left"

print("Padding side:", tokenizer.padding_side)

Padding side: left


In [16]:
print("Padding side:", tokenizer.padding_side)

Padding side: left


In [ ]:
# Batched G0 generation for all 1,000 evaluation examples

import torch
from tqdm.auto import tqdm

g0_model.eval()

BATCH_SIZE = 4
MAX_INPUT_LENGTH = 1024
MAX_NEW_TOKENS = 120

g0_predictions = []

for start in tqdm(
    range(0, len(gold_eval_set), BATCH_SIZE),
    desc="Generating G0 answers"
):

    # Get individual rows correctly from Hugging Face Dataset
    end = min(start + BATCH_SIZE, len(gold_eval_set))
    batch_rows = [gold_eval_set[i] for i in range(start, end)]

    prompts = []

    for row in batch_rows:

        context = row["context"]

        if isinstance(context, dict) and "contexts" in context:
            context_text = " ".join(context["contexts"])
        elif isinstance(context, list):
            context_text = " ".join(context)
        else:
            context_text = str(context)

        prompt = (
            f"### Question:\n{row['question']}\n\n"
            f"### Context:\n{context_text}\n\n"
            f"### Long Answer:\n"
        )

        prompts.append(prompt)

    # Tokenize the whole batch
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH
    )

    inputs = {
        k: v.to(g0_model.device)
        for k, v in inputs.items()
    }

    # Generate answers
    with torch.no_grad():
        output_ids = g0_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Decode each answer
    for j in range(len(batch_rows)):

        input_length = inputs["attention_mask"][j].sum().item()

        generated_ids = output_ids[j][input_length:]

        generated_answer = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        ).strip()

        g0_predictions.append(generated_answer)


print("\nGeneration complete!")
print("Number of G0 predictions:", len(g0_predictions))

Generating G0 answers:   0%|          | 0/250 [00:00<?, ?it/s]

In [11]:
# Generate G0 answers for all 1,000 evaluation examples

import torch
from tqdm.auto import tqdm

g0_model.eval()

g0_predictions = []

for i in tqdm(range(len(gold_eval_set))):

    row = gold_eval_set[i]

    # Prepare context
    context = row["context"]

    if isinstance(context, dict) and "contexts" in context:
        context_text = " ".join(context["contexts"])
    elif isinstance(context, list):
        context_text = " ".join(context)
    else:
        context_text = str(context)

    # Same prompt format
    prompt = (
        f"### Question:\n{row['question']}\n\n"
        f"### Context:\n{context_text}\n\n"
        f"### Long Answer:\n"
    )

    # Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    inputs = {k: v.to(g0_model.device) for k, v in inputs.items()}

    # Generate
    with torch.no_grad():
        output_ids = g0_model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Remove prompt tokens
    prompt_length = inputs["input_ids"].shape[1]

    generated_ids = output_ids[0][prompt_length:]

    generated_answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    g0_predictions.append(generated_answer)

print("Generation complete!")
print("Number of G0 predictions:", len(g0_predictions))

  0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [10]:
!pip install -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.0 MB/s eta 0:00:00


In [1]:
import bitsandbytes as bnb

print("bitsandbytes version:", bnb.__version__)

bitsandbytes version: 0.50.2
